# 01. Imports

In [13]:
import os
import glob
import pandas as pd
import numpy as np
from PIL import Image
from tqdm import tqdm
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as transforms
import torchvision.models as models

from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split

# 02. Config

In [ ]:
CONFIG = {
    "csv_path": "labels_synthetic_calibrated_with_path.csv",
    "base_dir": "",
    "checkpoint_dir": "/Saved Models/checkpoints/",
    "best_model_dir": "/Saved Models/best_models/",
    "image_col": "image_path",
    "concept_cols": ["NO", "NC", "C", "P"],
    "batch_size": 16,
    "lr": 3e-4,
    "epochs": 15,
    "img_size": 224,
    "device": "cuda" if torch.cuda.is_available() else "cpu"
}

# 03. Dataset

In [15]:
class CataractDataset(Dataset):
    def __init__(self, df, transform=None, base_dir=""):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.base_dir = base_dir  # ← ADD THIS

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Fix Windows backslashes, then prepend base_dir
        rel_path = row["relative_path"].replace("\\", "/")
        img_path = os.path.join(self.base_dir, rel_path)  # ← UPDATED

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        labels = torch.tensor([
            row["NO_pseudo"],
            row["NC_pseudo"],
            row["CO_pseudo"],
            row["PSC_pseudo"]
        ], dtype=torch.float32) / 5.0

        if row["source"] == "slit_lamp":
            mask = torch.tensor([1, 1, 1, 1], dtype=torch.float32)
        else:
            mask = torch.tensor([1, 1, 0, 0], dtype=torch.float32)

        return image, labels, mask

# 04. Transforms

In [16]:
train_tf = transforms.Compose([
    transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# 05. Model

In [17]:
class CBMModel(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.features = nn.Sequential(*list(backbone.children())[:-1])

        # Shared feature reduction only — stops at 128
        self.shared = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        self.concept_head = nn.Linear(128, 4)   # NO, NC, CO, PSC
        self.presence_head = nn.Linear(128, 1)  # cataract or not

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        shared = self.shared(x)
        concepts = self.concept_head(shared)
        presence = torch.sigmoid(self.presence_head(shared))
        return concepts, presence

# 06. Metrics

In [18]:
def compute_mae(pred, true):
    return torch.mean(torch.abs(pred - true)).item()

def compute_spearman(pred, true):
    pred = pred.detach().cpu().numpy()
    true = true.detach().cpu().numpy()

    corrs = []
    for i in range(pred.shape[1]):
        corr, _ = spearmanr(pred[:, i], true[:, i])
        corrs.append(corr)

    return np.mean(corrs)

# 07. Train & Validate

In [19]:
def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0

    for images, labels, mask in loader:
        images = images.to(device)
        labels = labels.to(device)
        mask   = mask.to(device)

        concepts, presence = model(images)
        loss = F.smooth_l1_loss(concepts * mask, labels * mask)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

def validate(model, loader, device):
    model.eval()

    total_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels, mask in loader:
            images = images.to(device)
            labels = labels.to(device)
            mask = mask.to(device)

            concepts, presence = model(images)
            loss = F.smooth_l1_loss(concepts * mask, labels * mask)

            total_loss += loss.item()

            all_preds.append(concepts)
            all_labels.append(labels)

    all_preds = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)

    mae = compute_mae(all_preds, all_labels)
    spearman = compute_spearman(all_preds, all_labels)

    return total_loss / len(loader), mae, spearman

# 08. Main Method

In [ ]:
def main():
    print("Loading dataset...")

    df = pd.read_csv(CONFIG["csv_path"])

    train_df, val_df = train_test_split(
        df, test_size=0.2, random_state=42, shuffle=True
    )

    print(f"Train size: {len(train_df)}")
    print(f"Val size:   {len(val_df)}")

    os.makedirs(CONFIG["checkpoint_dir"], exist_ok=True)
    os.makedirs(CONFIG["best_model_dir"],  exist_ok=True)

    train_ds = CataractDataset(train_df, train_tf, base_dir=CONFIG["base_dir"])
    val_ds   = CataractDataset(val_df,   val_tf,   base_dir=CONFIG["base_dir"])

    train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=CONFIG["batch_size"], shuffle=False)

    model     = CBMModel().to(CONFIG["device"])
    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"])

    best_loss   = float("inf")
    start_epoch = 0
    best_model_number = 0

    checkpoint_files = sorted(glob.glob(os.path.join(CONFIG["checkpoint_dir"], "checkpoint_epoch_*.pth")))
    
    # Find the next best model number
    best_models = sorted(glob.glob(os.path.join(CONFIG["best_model_dir"], "best_model_*.pth")))
    if best_models:
        best_model_number = len(best_models)
    
    best_model_path  = os.path.join(CONFIG["best_model_dir"], f"best_model_{best_model_number:02d}.pth")

    if checkpoint_files:
      # Try to load the most recent best model (if there are any)
      if best_models:
        recent_best = best_models[-1]
        try:
            best_ckpt = torch.load(recent_best, map_location=CONFIG["device"], weights_only=False)
            model.load_state_dict(best_ckpt["model_state"])
            best_loss = best_ckpt["val_loss"]
            print(f"Loaded best model weights (val_loss: {best_loss:.4f}, epoch: {best_ckpt['epoch']})")
        except RuntimeError:
            print("Old checkpoint architecture detected — starting fresh weights, keeping epoch counter only.")

      try:
        latest_ckpt = torch.load(checkpoint_files[-1], map_location=CONFIG["device"], weights_only=False)
        optimizer.load_state_dict(latest_ckpt["optimizer_state"])
        start_epoch = latest_ckpt["epoch"]
        print(f"Resuming optimizer from latest checkpoint (epoch: {start_epoch})")
      except RuntimeError:
        print("Optimizer state incompatible — starting optimizer fresh.")
        start_epoch = 0

    else:
      print("No checkpoints found — starting fresh training")

    end_epoch = start_epoch + CONFIG["epochs"]
    print(f"\nStarting training from epoch {start_epoch+1} to {end_epoch}...")
    total_start = time.time()

    for epoch in range(start_epoch, end_epoch):
        epoch_start = time.time()
        print(f"\nEpoch {epoch+1}")

        train_loss                      = train_epoch(model, train_loader, optimizer, CONFIG["device"])
        val_loss, val_mae, val_spearman = validate(model, val_loader, CONFIG["device"])

        epoch_end = time.time()
        epoch_time = epoch_end - epoch_start

        print(f"Train Loss: {train_loss:.4f}")
        print(f"Val Loss:   {val_loss:.4f} | MAE: {val_mae:.4f} | Spearman: {val_spearman:.4f}")
        print(f"Epoch time: {epoch_time:.2f} seconds")

        # ── Save checkpoint for this epoch ────────────────────────────────────
        checkpoint_path = os.path.join(
            CONFIG["checkpoint_dir"], f"checkpoint_epoch_{epoch+1:03d}.pth"
        )
        torch.save({
            "epoch":           epoch + 1,
            "model_state":     model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "val_loss":        val_loss,
            "val_mae":         val_mae,
            "val_spearman":    val_spearman,
            "best_loss":       best_loss
        }, checkpoint_path)
        print(f"Checkpoint saved → epoch {epoch+1:03d}")

        # ── Update best model if improved ─────────────────────────────────────
        if val_loss < best_loss:
            best_loss = val_loss
            torch.save({
                "epoch":        epoch + 1,
                "model_state":  model.state_dict(),
                "val_loss":     val_loss,
                "val_mae":      val_mae,
                "val_spearman": val_spearman
            }, best_model_path)
            print(f"Best model updated → {os.path.basename(best_model_path)} | val_loss: {val_loss:.4f} at epoch {epoch+1}")
            
            # Update filename for next potential best model
            best_model_number += 1
            best_model_path = os.path.join(CONFIG["best_model_dir"], f"best_model_{best_model_number:02d}.pth")

    total_end = time.time()
    total_time = total_end - total_start
    print(f"\nTraining complete! Epochs {start_epoch+1}–{end_epoch} done.")
    print(f"Best val loss so far: {best_loss:.4f}")
    print(f"Total training time: {total_time:.2f} seconds")

if __name__ == "__main__":
    main()


Loading dataset...
Train size: 16291
Val size:   4073
Loaded best model weights (val_loss: 0.0016, epoch: 3)
Resuming optimizer from latest checkpoint (epoch: 4)

Starting training from epoch 5 to 19...

Epoch 5
